In [110]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.__version__)

2.14.1+cu130


Load the dataset from kaggle directly

In [111]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("gurudathg/crop-yield-prediction-using-soil-and-weather")

Using Colab cache for faster access to the 'crop-yield-prediction-using-soil-and-weather' dataset.


Train Test Split and normalized the Input

In [112]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv(path+'/Crop Yiled with Soil and Weather.csv')
x = df['N']
y = df['yeild']
print(x.shape, y.shape)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Normalize X_train (and apply the same mean/std to X_test later)
X_mean = X_train.mean()
X_std = X_train.std()
X_train = (X_train - X_mean) / X_std


(2596,) (2596,)


Load the x and y into the CUDA

In [113]:
X_train = torch.tensor(
    X_train.values,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
)

X_train = X_train.to(device)
y_train = y_train.to(device)

b = torch.zeros(1, device=device, requires_grad=True)
w = torch.zeros(1, device=device, requires_grad=True)

In [114]:
loss = ((w*X_train+b-y_train)**2).mean()

loss.backward()

In [115]:
print("Loss:", loss.item())

print("Weight gradient:", w.grad)

print("Bias gradient:", b.grad)

Loss: 76.64830017089844
Weight gradient: tensor([-3.4017], device='cuda:0')
Bias gradient: tensor([-17.0753], device='cuda:0')


Using the Adam optimizer

In [116]:
optimizer = torch.optim.Adam([w, b], lr=1e-2)

Doing 2000 epoch

In [117]:
for epoch in range(2000):

    optimizer.zero_grad()
    loss = ((w*X_train+b-y_train)**2).mean()

    loss.backward()

    optimizer.step()

    if (epoch+1) % 100 == 0:
        print(epoch+1, loss.item())

100 58.81690979003906
200 45.13153839111328
300 34.37137222290039
400 25.79769515991211
500 19.044275283813477
600 13.826849937438965
700 9.884529113769531
800 6.977812767028809
900 4.891526222229004
1000 3.4373152256011963
1100 2.4553143978118896
1200 1.8143593072891235
1300 1.4109253883361816
1400 1.1665712594985962
1500 1.0244524478912354
1600 0.9452393651008606
1700 0.9030095338821411
1800 0.8815228939056396
1900 0.8711092472076416
2000 0.8663115501403809


In [118]:
print("Weight:", w.item())
print("Bias:", b.item())

Weight: 1.7016888856887817
Bias: 8.478860855102539


Testing the model using the test set

In [119]:
x_test = torch.tensor(X_test.values, dtype=torch.float32, device=device)
y_test = torch.tensor(y_test.values, dtype=torch.float32, device=device)

x_test = (x_test - X_mean) / X_std

with torch.no_grad():
    y_pred = w * x_test + b

test_loss = ((y_pred - y_test)**2).mean()
test_rmse = torch.sqrt(test_loss)

print("Test Loss:", test_loss.item())
print("Test RMSE:", test_rmse.item())
print("Actual values:", y_test[:5])
print("Predicted values:", y_pred[:5])

Test Loss: 0.889833390712738
Test RMSE: 0.9433097839355469
Actual values: tensor([ 8.0555,  8.4662,  6.4854, 10.1934,  6.3988], device='cuda:0')
Predicted values: tensor([ 8.5675,  8.5921,  6.5548, 10.3665,  7.2967], device='cuda:0')
